In [ ]:
pip install azure-ai-textanalytics azure-core python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

# Load environment variables from .env file
load_dotenv()

endpoint = os.getenv("AZURE_LANGUAGE_ENDPOINT")
key = os.getenv("AZURE_LANGUAGE_KEY")

# Initialize the client
client = TextAnalyticsClient(
    endpoint=endpoint, 
    credential=AzureKeyCredential(key)
)

# Test with a simple sentiment analysis call
documents = [
    "The service was excellent.",
    "The delivery was very slow.",
    "The product is acceptable."
]

results = client.analyze_sentiment(documents=documents)

for text, result in zip(documents, results):
    if not result.is_error:
        print(text, "->", result.sentiment)

In [ ]:
import os
from dotenv import load_dotenv

from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError

# 1. Read Azure credentials from environment variables
load_dotenv()
endpoint = os.environ["VISION_ENDPOINT"]
key = os.environ["VISION_KEY"]

# 2. Connect to the Azure Computer Vision resource
client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

# 3. Specify the local image path
image_path = "sample.jpg"

try:
    # 4. Read image bytes
    with open(image_path, "rb") as image_file:
        image_data = image_file.read()

    # 5. Send the image to Azure for Analysis
    result = client.analyze(
        image_data=image_data,
        visual_features=[
            VisualFeatures.CAPTION,
            VisualFeatures.READ
        ],
        gender_neutral_caption=True
    )

    # 6. Output Caption Results
    print("\n--- Image Recognition Results ---")

    if result.caption:
        print("\nImage Description:")
        print(result.caption.text)
        print(f"Confidence: {result.caption.confidence:.2%}")

    # 7. Extract text from the image using OCR
    if result.read:
        print("\nText Detected in Image:")

        for block in result.read.blocks:
            for line in block.lines:
                print(line.text)

    print("\nImage analysis completed successfully.")

except FileNotFoundError:
    print(f"Image file not found: {image_path}")
except HttpResponseError as e:
    print(f"Azure API call failed: {e}")

In [ ]:
# Azure Immersive Reader

In [ ]:
%pip install flask requests python-dotenv

In [1]:
import os
import requests
from flask import Flask, jsonify, render_template_string, request
from dotenv import load_dotenv

# Load environment variables (override=True ensures updates to .env are picked up on re-runs)
load_dotenv(override=True)

app = Flask(__name__)

# Azure credentials
TENANT_ID = os.environ["TENANT_ID"]
CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]
SUBDOMAIN = os.environ["SUBDOMAIN"]

# HTML frontend
HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Azure Immersive Reader</title>
    <script src="https://ircdname.azureedge.net/immersivereadersdk/immersive-reader-sdk.1.4.0.js"></script>
    <style>
        body {
            font-family: Arial;
            margin: 40px;
            background: #f5f5f5;
        }
        .container {
            background: white;
            padding: 30px;
            border-radius: 10px;
            max-width: 800px;
            margin: auto;
        }
        textarea {
            width: 100%;
            height: 180px;
            padding: 10px;
            font-size: 16px;
            box-sizing: border-box;
        }
        button {
            padding: 12px 20px;
            background: #0078d4;
            color: white;
            border: none;
            cursor: pointer;
            margin-top: 15px;
            border-radius: 5px;
        }
        button:hover {
            background: #005a9e;
        }
        #status {
            margin-top: 15px;
        }
    </style>
</head>
<body>
<div class="container">
    <h1>Azure Immersive Reader</h1>
    <p>Enter some text below and launch Microsoft Immersive Reader.</p>
    <textarea id="reading-content">Machine learning enables computers to learn patterns from data and make predictions.

Artificial intelligence is transforming industries by automating tasks and improving decision-making.

Python is widely used for machine learning and artificial intelligence development.</textarea>
    <br>
    <button onclick="launchReader()">Open Immersive Reader</button>
    <p id="status"></p>
</div>

<script>
async function launchReader() {
    const status = document.getElementById("status");
    status.innerText = "Obtaining Azure access token...";
    try {
        const response = await fetch("/token", { cache: "no-store" });
        if (!response.ok) {
            throw new Error("Authentication failed");
        }
        const auth = await response.json();
        const readingText = document.getElementById("reading-content").value;

        const content = {
            title: "My Reading Content",
            chunks: [
                {
                    content: readingText,
                    lang: "en"
                }
            ]
        };

        status.innerText = "Launching Immersive Reader...";
        await ImmersiveReader.launchAsync(auth.token, auth.subdomain, content);
        status.innerText = "Immersive Reader closed.";
    } catch (error) {
        status.innerText = "Error: " + error.message;
        console.error(error);
    }
}
</script>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(HTML_PAGE)

@app.route("/token")
def get_token():
    token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "scope": "https://cognitiveservices.azure.com/.default",
        "grant_type": "client_credentials"
    }

    try:
        response = requests.post(token_url, data=payload, timeout=20)
        response.raise_for_status()
        token_data = response.json()
        return jsonify({
            "token": token_data["access_token"],
            "subdomain": SUBDOMAIN
        })
    except requests.RequestException as e:
        error_detail = e.response.text if e.response is not None else str(e)
        print("Azure error:", error_detail)
        return jsonify({"error": "Unable to authenticate with Azure", "details": error_detail}), 502

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [10/Nov/2026 13:51:44] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 13:51:49] "GET /token HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 13:51:55] "GET /favicon.ico HTTP/1.1" 404 -
